In [102]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn import impute

In [103]:
dataset = pd.read_csv("../data_set/Data.csv")
dataset.head()

,Country,Age,Salary,Purchased
0,France,44.0,72000.0,No
1,Spain,27.0,48000.0,Yes
2,Germany,30.0,54000.0,No
3,Spain,38.0,61000.0,No
4,Germany,40.0,NaN,Yes


In [104]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Country    10 non-null     str    
 1   Age        9 non-null      float64
 2   Salary     9 non-null      float64
 3   Purchased  10 non-null     str    
dtypes: float64(2), str(2)
memory usage: 452.0 bytes


In [105]:
# .values: Converts the selected column from a pandas Series into a NumPy array.
x = dataset.iloc[:, :-1].values
y = dataset.iloc[:, 3].values
print("---------- Feature Matrix : X -----------")
print(X)
print("---------- Feature Matrix : y -----------")
print(y)

---------- Feature Matrix : X -----------
    Age   Salary  Country
0  44.0  72000.0   France
1  27.0  48000.0    Spain
2  30.0  54000.0  Germany
3  38.0  61000.0    Spain
4  40.0  61000.0  Germany
5  35.0  58000.0   France
6  38.0  52000.0    Spain
7  48.0  79000.0   France
8  50.0  83000.0  Germany
9  37.0  67000.0   France
---------- Feature Matrix : y -----------
<StringArray>
['No', 'Yes', 'No', 'No', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes']
Length: 10, dtype: str


In [106]:
X = dataset.iloc[:, :-1]
y = dataset.iloc[:, 3]
print("---------- Feature Matrix : X -----------")
print(X)
print("---------- Feature Matrix : y -----------")
print(y)

---------- Feature Matrix : X -----------
   Country   Age   Salary
0   France  44.0  72000.0
1    Spain  27.0  48000.0
2  Germany  30.0  54000.0
3    Spain  38.0  61000.0
4  Germany  40.0      NaN
5   France  35.0  58000.0
6    Spain   NaN  52000.0
7   France  48.0  79000.0
8  Germany  50.0  83000.0
9   France  37.0  67000.0
---------- Feature Matrix : y -----------
0     No
1    Yes
2     No
3     No
4    Yes
5    Yes
6     No
7    Yes
8     No
9    Yes
Name: Purchased, dtype: str


## Taking care of missing data

In [107]:
from sklearn.impute import SimpleImputer

# seperting numerical comumns
num_cols = X.select_dtypes(include=[np.number]).columns
print(num_cols)


Index(['Age', 'Salary'], dtype='str')


In [108]:
# Impute the whole matrix at once

imputer = SimpleImputer(missing_values=np.nan, strategy="median").set_output(transform="pandas")
X[num_cols] = imputer.fit_transform(X[num_cols])
print("---------- Feature Matrix : X -----------")
print(X)

---------- Feature Matrix : X -----------
   Country   Age   Salary
0   France  44.0  72000.0
1    Spain  27.0  48000.0
2  Germany  30.0  54000.0
3    Spain  38.0  61000.0
4  Germany  40.0  61000.0
5   France  35.0  58000.0
6    Spain  38.0  52000.0
7   France  48.0  79000.0
8  Germany  50.0  83000.0
9   France  37.0  67000.0


In [109]:
# Use ColumnTransformer for Mixed Datasets

from sklearn.compose import ColumnTransformer

ct = ColumnTransformer(
    transformers = [
        ("age", SimpleImputer(strategy="mean"), ["Age",]),
        ("salary", SimpleImputer(strategy="median"), ["Salary",],),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False, # Stops scikit-learn from prefixing 'step__' to columns
).set_output(transform='pandas')

X = ct.fit_transform(X)
print("---------- Feature Matrix : X -----------")
print(X)

---------- Feature Matrix : X -----------
    Age   Salary  Country
0  44.0  72000.0   France
1  27.0  48000.0    Spain
2  30.0  54000.0  Germany
3  38.0  61000.0    Spain
4  40.0  61000.0  Germany
5  35.0  58000.0   France
6  38.0  52000.0    Spain
7  48.0  79000.0   France
8  50.0  83000.0  Germany
9  37.0  67000.0   France


In [110]:
# Manually Hangling Missing Values
Age = dataset.Age
Age = dataset["Age"]
Salary = dataset["Salary"]
dataset.Salary.median()
dataset.Age.median()
dataset.Age = dataset.Age.fillna(dataset.Age.median())
dataset["Age"] = dataset.Age.fillna(dataset.Age.median())
dataset.Salary = dataset.Salary.fillna(dataset.Salary.median())

## Label Encoding
- Label Encoding is a preprocessing technique that converts categorical (text) values into numbers.

In [111]:
from sklearn.preprocessing import LabelEncoder

print("----- Before Label Encoding -----\n")
print(X)

labelencoder_X = LabelEncoder()
X_lab_enc = X.copy()
X_lab_enc["Country"] = labelencoder_X.fit_transform(X['Country'])
print("----- After Label Encoding -----\n")
print(X_lab_enc)

----- Before Label Encoding -----

    Age   Salary  Country
0  44.0  72000.0   France
1  27.0  48000.0    Spain
2  30.0  54000.0  Germany
3  38.0  61000.0    Spain
4  40.0  61000.0  Germany
5  35.0  58000.0   France
6  38.0  52000.0    Spain
7  48.0  79000.0   France
8  50.0  83000.0  Germany
9  37.0  67000.0   France
----- After Label Encoding -----

    Age   Salary  Country
0  44.0  72000.0        0
1  27.0  48000.0        2
2  30.0  54000.0        1
3  38.0  61000.0        2
4  40.0  61000.0        1
5  35.0  58000.0        0
6  38.0  52000.0        2
7  48.0  79000.0        0
8  50.0  83000.0        1
9  37.0  67000.0        0


In [112]:
# Label Encoding : Features Vector y
print("----- Before Label Encoding -----\n")
print(y)
labelencoder_y = LabelEncoder()
y = labelencoder_y.fit_transform(y)
print("----- After Label Encoding -----\n")
print(y)

----- Before Label Encoding -----

0     No
1    Yes
2     No
3     No
4    Yes
5    Yes
6     No
7    Yes
8     No
9    Yes
Name: Purchased, dtype: str
----- After Label Encoding -----

[0 1 0 0 1 1 0 1 0 1]


## OneHotEncoer
- OneHotEncoder is a preprocessing technique that converts categorical variables into multiple binary (0 or 1) columns.
- It is commonly used for nominal categories (categories with no natural order).

In [121]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# creates a ColumnTransformer that applies One-Hot Encoding only to the first column (Country) and leaves all other columns unchanged.
# sparse_output=False Prevents compatibility issues and forces a dense matrix output.
country = ColumnTransformer(
    transformers=[("Country", OneHotEncoder(sparse_output=False), ["Country",])], 
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

X_ohe = country.fit_transform(X)
print(X_ohe)

   Country_France  Country_Germany  Country_Spain   Age   Salary
0             1.0              0.0            0.0  44.0  72000.0
1             0.0              0.0            1.0  27.0  48000.0
2             0.0              1.0            0.0  30.0  54000.0
3             0.0              0.0            1.0  38.0  61000.0
4             0.0              1.0            0.0  40.0  61000.0
5             1.0              0.0            0.0  35.0  58000.0
6             0.0              0.0            1.0  38.0  52000.0
7             1.0              0.0            0.0  48.0  79000.0
8             0.0              1.0            0.0  50.0  83000.0
9             1.0              0.0            0.0  37.0  67000.0


## Feature scaling is the process of changing the range or scale of numerical features so that they are comparable.

In [123]:
# Feature Scaling : Feature Matrix - X
from sklearn.preprocessing import StandardScaler

num_cols = ["Age", "Salary"]
sc_X = StandardScaler().set_output(transform="pandas")
X_ss = sc_X.fit_transform(X[num_cols])
print(X_ss)

        Age    Salary
0  0.769734  0.772568
1 -1.699225 -1.408800
2 -1.263526 -0.863458
3 -0.101663 -0.227226
4  0.188803 -0.227226
5 -0.537362 -0.499897
6 -0.101663 -1.045239
7  1.350666  1.408800
8  1.641132  1.772361
9 -0.246896  0.318116


In [126]:
X_ohe[num_cols] = X_ss[num_cols]
print(X_ohe)
X = X_ohe

   Country_France  Country_Germany  Country_Spain       Age    Salary
0             1.0              0.0            0.0  0.769734  0.772568
1             0.0              0.0            1.0 -1.699225 -1.408800
2             0.0              1.0            0.0 -1.263526 -0.863458
3             0.0              0.0            1.0 -0.101663 -0.227226
4             0.0              1.0            0.0  0.188803 -0.227226
5             1.0              0.0            0.0 -0.537362 -0.499897
6             0.0              0.0            1.0 -0.101663 -1.045239
7             1.0              0.0            0.0  1.350666  1.408800
8             0.0              1.0            0.0  1.641132  1.772361
9             1.0              0.0            0.0 -0.246896  0.318116


In [127]:
# train and test data split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=0)

In [128]:
print(X_train)
print(y_train)
print(X_test)
print(y_test)

   Country_France  Country_Germany  Country_Spain       Age    Salary
1             0.0              0.0            1.0 -1.699225 -1.408800
6             0.0              0.0            1.0 -0.101663 -1.045239
7             1.0              0.0            0.0  1.350666  1.408800
3             0.0              0.0            1.0 -0.101663 -0.227226
0             1.0              0.0            0.0  0.769734  0.772568
5             1.0              0.0            0.0 -0.537362 -0.499897
[1 0 1 0 0 1]
   Country_France  Country_Germany  Country_Spain       Age    Salary
2             0.0              1.0            0.0 -1.263526 -0.863458
8             0.0              1.0            0.0  1.641132  1.772361
4             0.0              1.0            0.0  0.188803 -0.227226
9             1.0              0.0            0.0 -0.246896  0.318116
[0 0 1 1]


## Logistic Regression

In [133]:
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(random_state=0)
classifier.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",0
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver

In [143]:
y_pred = classifier.predict(X_test)
print(y_pred)

[1 0 0 1]


In [135]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix : \n", cm)

Confusion Matrix : 
 [[1 1]
 [1 1]]


In [137]:
from sklearn.metrics import accuracy_score

print("Accuracy Score : ", accuracy_score(y_test, y_pred))

Accuracy Score :  0.5


## SVC
- SVC stands for Support Vector Classifier. It is the classification implementation of the Support Vector Machine (SVM) algorithm in scikit-learn.

In [138]:
from sklearn.svm import SVC

svm = SVC()
svm.fit(X_train, y_train)

y_pred = svm.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix : \n", cm)

print(
    "Accuracy of SVM classifier on training set: {:.2f}".format(
        svm.score(X_train, y_train)
    )
)
print(
    "Accuracy of SVM classifier on test set: {:.2f}".format(svm.score(X_test, y_test))
)

Confusion Matrix : 
 [[0 2]
 [1 1]]
Accuracy of SVM classifier on training set: 0.83
Accuracy of SVM classifier on test set: 0.25


## Decision Tree Classifier

In [139]:
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier()
dtc.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [140]:
print(
    "Accuracy of Decision Tree classifier on training set: {:.2f}".format(
        dtc.score(X_train, y_train)
    )
)
print(
    "Accuracy of Decision Tree classifier on test set: {:.2f}".format(
        dtc.score(X_test, y_test)
    )
)

Accuracy of Decision Tree classifier on training set: 1.00
Accuracy of Decision Tree classifier on test set: 0.00


In [141]:
dtc.predict(X_test)

array([1, 1, 0, 0])

In [145]:
dtc.predict(X_test.iloc[[0]])

array([1])